In [0]:
# Bibliotecas que serão usadas no notebook
import requests, os 

from datetime import datetime
import pandas as pd

# Função para gerar URLs de anos futuros
def gerar_url(year):
    base_url = "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/baseDados/celularesSub/CelularesSubtraidos_"
    return f"{base_url}{year}.xlsx"

# Lista de anos disponíveis.
anos = list(range(2025, datetime.now().year + 1))
urls = [gerar_url(ano) for ano in anos]

# Path do volume
volume_path = "/Volumes/workspace/treinamentos/projeto_celulares_sp/"


In [0]:
#Função para fazer o download
def download(url, volume_path):
    response = requests.get(url)

    if response.status_code == 200:
        with open(volume_path, 'wb') as file:
            file.write(response.content)
        print(f" Download Concluído para {volume_path}")
        return True
    else:
        print(f" Erro ao fazer download: {response.status_code}")
        return False

In [0]:
def baixar_planilhas(urls, volume_path, mes_limite=4):

    hoje = datetime.now()
    for url in urls:
        # Extrai o ano do URL
        ano = int(url.split('_')[-1].rstrip('.xlsx'))
        # Limite de download ate o mês 3 do ano subsequente
        limite = datetime(ano + 1, mes_limite, 1)
        destino_arquivo = f"{volume_path}celulares_{ano}.xlsx"

        # Sempre sobrescreve até o limite do ano subsequente
        if hoje <= limite:
            download(url, destino_arquivo)
            print(f" Arquivo {ano} baixado")
        else:
            if not os.path.exists(destino_arquivo):
                download(url, destino_arquivo)
                print(f" Arquivo {ano}ainda não existia, feito download")
            else:
                print(f" Arquivo {ano} já baixado e acima do limite")

In [0]:
#Instala o openpyxl para ler o excel
%pip install openpyxl

In [0]:
def converter_para_parquet(anos, volume_path):

     for ano in anos:
        xlsx = f"{volume_path}celulares_{ano}.xlsx"
        parquet = f"{volume_path}celulares_{ano}.parquet"
        #Conversão para Parquet e Str
        try:
            df = pd.read_excel(xlsx, sheet_name=f"CELULAR_{ano}")
            df = df.astype(str)
            df.to_parquet(parquet)
            print(f" convertido para Parquet: {parquet}")
        except Exception as e:
            print(f" erro ao processar {xlsx}: {str(e)}")

In [0]:
# Função para remover todos os arquivos em formato excel
def remover_arquivos_excel(volume_path):

    files = dbutils.fs.ls(volume_path)
    for f in files:
        if f.name.endswith('.xlsx'):
            dbutils.fs.rm(f.path)
            print(f" Arquivo {f.path} removido")

In [0]:
# Executa a ingestão
baixar_planilhas(urls, volume_path)
converter_para_parquet(anos, volume_path)
remover_arquivos_excel(volume_path)